# Pathology Hub Governed Cleanup v10.5 — manifest/API proof + strict figure cleanup

Use this after v10.4. It verifies the live promoted metadata, cleans stale manifest summaries that can still display old lecture/textbook junk tags, runs authenticated API proof using the Colab secret `X-API-Key`, and performs a stricter textbook figure-dimension filter.

Promotion is enabled by default. It backs up live manifests/figure map before replacing them.

In [ ]:
# Cell 1 — Config
import os, json, re, csv, time, shutil, hashlib, subprocess, sqlite3, math, tempfile
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

try:
    from google.colab import auth, userdata, files
    IN_COLAB=True
except Exception:
    IN_COLAB=False

PROJECT_ID = "pathology-annotation-project"
SERVICE = "pathology-hub-v04"
REGION = "us-central1"
HEALTH_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app/health"
API_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app/evidence/search"
API_KEY_SECRET_NAME = "X-API-Key"

PROMOTION_MODE = "backup_replace_live"   # enabled by default per request
RESTART_CLOUD_RUN_AFTER_PROMOTION = True
RUN_API_PROOF = True
STRICT_FIGURE_FILTER = True
DIMENSION_CHECK_IMAGES = True              # attempts to read actual image dimensions when metadata lacks dimensions
MAX_DIMENSION_CHECK = None                 # set small integer for debugging; None = all public figure rows
FIGURE_DOWNLOAD_TIMEOUT = 12
FIGURE_WORKERS = 16

# Stricter but reversible junk rules. These affect served derivative figure-map records only, not raw PDFs.
MIN_DIMENSION_EXCLUDE = 80
MIN_AREA_EXCLUDE = 9000
MAX_ASPECT_RATIO_EXCLUDE = 10.0
CAPTION_OR_LABEL_KEYWORDS = re.compile(r"\b(header|footer|copyright|chapter title|section title|caption only|label only)\b", re.I)

RUN_TS = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK = Path(f"/content/pathology_hub_governed_cleanup_v10_5_{RUN_TS}") if IN_COLAB else Path('/mnt/data/pathology_hub_governed_cleanup_v10_5_local')
DATA = WORK/'data'; OUT = WORK/'output'; STAGED = WORK/'staged'; BACKUP_LOCAL = WORK/'backup_local'
for p in [DATA,OUT,STAGED,BACKUP_LOCAL]: p.mkdir(parents=True, exist_ok=True)

GCS = {
    'tb_docstore': 'gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl',
    'tb_manifest': 'gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json',
    'pathout_docstore': 'gs://pathology_hub/03_indexes/pathology_outlines/pathout_allsite_v0_1/vector_ap_diagnostic_v1/pathout_ap_diagnostic_vector_docstore.jsonl',
    'pathout_manifest': 'gs://pathology_hub/03_indexes/pathology_outlines/pathout_allsite_v0_1/vector_ap_diagnostic_v1/pathout_ap_diagnostic_vector_manifest.json',
    'lec_docstore': 'gs://pathology_hub/03_indexes/lectures/vector_STRICT_CYTO_v9/lecture_timecoded_vector_docstore_STRICT_CYTO_v9.jsonl',
    'lec_manifest': 'gs://pathology_hub/03_indexes/lectures/vector_STRICT_CYTO_v9/lecture_timecoded_vector_manifest_STRICT_CYTO_v9.json',
    'figure_map': 'gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl',
    'stage_root': 'gs://pathology_hub/02_normalized/tags/governance/v10_5',
    'audit_root': 'gs://pathology_hub/06_audits/tags/governance/v10_5',
    'backup_root': f'gs://pathology_hub/99_backups/governance_v10_5/{RUN_TS}',
}

print('WORK:', WORK)
print(json.dumps(GCS, indent=2))


In [ ]:
# Cell 2 — Auth/helpers
if IN_COLAB:
    auth.authenticate_user()

def now(): return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sh(cmd, check=True, timeout=900):
    print(f"\n[{now()}] START: {cmd}", flush=True)
    t0=time.time()
    p=subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    print(p.stdout[-12000:], flush=True)
    print(f"[{now()}] DONE rc={p.returncode} elapsed={time.time()-t0:.1f}s", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed rc={p.returncode}: {cmd}")
    return p

sh(f"gcloud config set project {PROJECT_ID}")

try:
    API_KEY = userdata.get(API_KEY_SECRET_NAME) if IN_COLAB else os.environ.get(API_KEY_SECRET_NAME)
except Exception:
    API_KEY = os.environ.get(API_KEY_SECRET_NAME)
print('API key loaded from Colab secret:', bool(API_KEY))

def gcp(src, dst, check=True):
    return sh(f"gcloud storage cp {src} {dst}", check=check, timeout=1800)

def gcp_ls(uri, check=True):
    return sh(f"gcloud storage ls -l {uri}", check=check, timeout=300)

def iter_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if line:
                try: yield json.loads(line)
                except Exception: continue

def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf-8') as f: json.dump(obj,f,indent=2,ensure_ascii=False)

def write_jsonl(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False,separators=(',',':'))+'\n')


In [ ]:
# Cell 3 — Download live metadata/manifest/figure map
local = {
    'tb_docstore': DATA/'textbook_lean_vector_docstore_live.jsonl',
    'tb_manifest': DATA/'textbook_lean_vector_manifest_live.json',
    'pathout_docstore': DATA/'pathout_ap_diagnostic_vector_docstore_live.jsonl',
    'pathout_manifest': DATA/'pathout_ap_diagnostic_vector_manifest_live.json',
    'lec_docstore': DATA/'lecture_timecoded_vector_docstore_STRICT_CYTO_live.jsonl',
    'lec_manifest': DATA/'lecture_timecoded_vector_manifest_STRICT_CYTO_live.json',
    'figure_map': DATA/'textbook_figure_web_map_live.jsonl',
}
for k,p in local.items():
    gcp(GCS[k], p)
print('Downloaded live files:')
for k,p in local.items(): print(k, p, p.stat().st_size)


In [ ]:
# Cell 4 — Verify live primary_tag fields are clean and recompute manifest summaries
FORBIDDEN_TAG_RE = re.compile(r"(::Lectures::|::Textbooks::|Digital_Pathology_Slide|Benign_Cystic_Neck_Mass_Case_01|::Error$|_Lecture_|YT_[A-Za-z]|Slide_\d+|Page_\d+|\bseconds_rule\b)", re.I)
WEAK_LEAF_RE = re.compile(r"^(slide|page|figure|fig|error|unknown|unidentified|none|null|case_?\d*|\d+)$", re.I)

def normalize_tag(t): return str(t or '').strip()
def is_unmapped(t): return normalize_tag(t) in {'','__UNMAPPED__','UNMAPPED','None','null'}
def is_forbidden(t):
    t=normalize_tag(t)
    if is_unmapped(t): return False
    if FORBIDDEN_TAG_RE.search(t): return True
    leaf=t.split('::')[-1]
    if WEAK_LEAF_RE.search(leaf): return True
    if sum(bool(re.search(r'\d{3,}', p)) for p in t.split('::')) >= 2: return True
    return False

def record_id(row, idx):
    for k in ['chunk_id','record_id','id','page_id','url','source_url']:
        if row.get(k): return str(row.get(k))
    return f'row:{idx}'

def scan(path, source):
    counts=Counter(); tags=Counter(); statuses=Counter(); examples_bad=[]; examples_unmapped=[]; examples=[]
    for idx,row in enumerate(iter_jsonl(path),1):
        t=normalize_tag(row.get('primary_tag'))
        tg=normalize_tag(row.get('primary_tag_governed'))
        status=row.get('tag_governance_status') or ''
        counts['total']+=1
        if is_unmapped(t):
            counts['primary_tag_unmapped']+=1
            if len(examples_unmapped)<20: examples_unmapped.append({'row':idx,'id':record_id(row,idx),'primary_tag':t,'governed':tg,'status':status})
        else:
            counts['primary_tag_mapped']+=1
            tags[t]+=1
        if tg:
            if is_unmapped(tg): counts['primary_tag_governed_unmapped']+=1
            else: counts['primary_tag_governed_mapped']+=1
        if is_forbidden(t):
            counts['forbidden_primary_tag']+=1
            if len(examples_bad)<50: examples_bad.append({'row':idx,'id':record_id(row,idx),'primary_tag':t,'governed':tg,'status':status})
        statuses[status]+=1
        if len(examples)<5: examples.append({'id':record_id(row,idx),'primary_tag':t,'status':status})
    return {'source':source,'path':str(path),'counts':dict(counts),'status_counts':dict(statuses),'unique_primary_tags':len(tags),'top_primary_tags':tags.most_common(100),'examples_forbidden':examples_bad,'examples_unmapped':examples_unmapped,'examples':examples}

scans = {
    'textbooks': scan(local['tb_docstore'],'textbooks'),
    'pathout': scan(local['pathout_docstore'],'pathout'),
    'lectures': scan(local['lec_docstore'],'lectures'),
}
write_json(OUT/'LIVE_GOVERNED_METADATA_SCAN_v10_5.json', scans)
print(json.dumps({k:{'counts':v['counts'],'unique_primary_tags':v['unique_primary_tags']} for k,v in scans.items()}, indent=2)[:4000])

fatal=[]
for k,v in scans.items():
    if v['counts'].get('forbidden_primary_tag',0): fatal.append(f"{k}: forbidden_primary_tag={v['counts'].get('forbidden_primary_tag')}")
# unmapped primary_tag rows are allowed only if they are intentionally not tag-indexed; but they may appear in API results, so warn hard.
if fatal:
    write_json(OUT/'LIVE_GOVERNED_METADATA_SCAN_FAILURE_v10_5.json', {'fatal':fatal,'scans':scans})
    raise RuntimeError('Live governed primary_tag still contains forbidden junk: '+ '; '.join(fatal))


In [ ]:
# Cell 5 — Patch stale manifest summaries from actual live docstores
manifest_specs = {
    'textbooks': {'in': local['tb_manifest'], 'out': STAGED/'textbook_lean_vector_manifest_v10_5_clean_summary.json', 'scan': scans['textbooks'], 'schema':'textbook_lean_vector_manifest.v10_5_governed_summary_cleanup'},
    'pathout': {'in': local['pathout_manifest'], 'out': STAGED/'pathout_ap_diagnostic_vector_manifest_v10_5_clean_summary.json', 'scan': scans['pathout'], 'schema':'pathout_ap_diagnostic_vector_manifest.v10_5_governed_summary_cleanup'},
    'lectures': {'in': local['lec_manifest'], 'out': STAGED/'lecture_timecoded_vector_manifest_STRICT_CYTO_v10_5_clean_summary.json', 'scan': scans['lectures'], 'schema':'lecture_timecoded_vector_manifest.v10_5_governed_summary_cleanup'},
}
for source,spec in manifest_specs.items():
    m=json.load(open(spec['in']))
    sc=spec['scan']
    # Preserve manifest content, but remove stale nested v9/v10.3 tag summaries and replace with governed fields.
    m['schema_version'] = spec['schema']
    m['tag_governance_version'] = 'v10_5_manifest_api_figure_strict_cleanup'
    m['tag_governance_generated_at_utc'] = now()
    m['record_count'] = sc['counts'].get('total') or m.get('record_count')
    m['primary_tag_unique_count'] = sc['unique_primary_tags']
    m['unmapped_count_docstore'] = sc['counts'].get('primary_tag_unmapped',0)
    m['forbidden_primary_tag_count_docstore'] = sc['counts'].get('forbidden_primary_tag',0)
    m['governance_status_counts_docstore'] = sc['status_counts']
    m['top_primary_tags'] = sc['top_primary_tags'][:100]
    # Clean stale nested structures known to display old junk.
    if isinstance(m.get('counts'), dict):
        m['counts']['tag_status_counts'] = sc['status_counts']
        m['counts']['primary_tag_unique_count'] = sc['unique_primary_tags']
        m['counts']['top_primary_tags'] = sc['top_primary_tags'][:100]
        m['counts']['__UNMAPPED__'] = sc['counts'].get('primary_tag_unmapped',0)
    m['notes'] = list(dict.fromkeys((m.get('notes') or []) + [
        'v10.5 recomputed manifest tag summaries from live governed docstore primary_tag fields.',
        'Stale generated lecture/textbook artifact tag summaries removed/replaced.',
    ]))
    write_json(spec['out'], m)
    print(source, 'manifest patched ->', spec['out'])


In [ ]:
# Cell 6 — Strict figure-dimension filtering
# This filters the served figure web map only. It does not delete raw PDFs or raw extracted images.
import concurrent.futures
from io import BytesIO
try:
    from PIL import Image, ImageFile
    ImageFile.LOAD_TRUNCATED_IMAGES = True
except Exception as e:
    print('PIL import failed:', e)
    Image = None
try:
    import requests
except Exception as e:
    print('requests import failed:', e)
    requests = None

def get_dim_from_record(row):
    keys_w=['width','image_width','w','bbox_width','pixel_width']
    keys_h=['height','image_height','h','bbox_height','pixel_height']
    w=h=None
    for k in keys_w:
        if row.get(k) not in [None,'']:
            try: w=int(float(row.get(k))); break
            except: pass
    for k in keys_h:
        if row.get(k) not in [None,'']:
            try: h=int(float(row.get(k))); break
            except: pass
    return w,h

def image_url_for(row):
    for k in ['image_url','figure_url','public_url','url','web_url']:
        v=row.get(k)
        if isinstance(v,str) and v.startswith('http'): return v
    return None

def read_http_dimensions(url):
    if not requests or not Image: return None
    try:
        r=requests.get(url, timeout=FIGURE_DOWNLOAD_TIMEOUT, stream=True)
        r.raise_for_status()
        data=BytesIO()
        # Usually enough for dimensions; fall back until parser opens.
        for i,chunk in enumerate(r.iter_content(8192)):
            if not chunk: break
            data.write(chunk)
            try:
                im=Image.open(BytesIO(data.getvalue()))
                return im.size
            except Exception:
                pass
            if data.tell() > 512_000: break
        im=Image.open(BytesIO(data.getvalue()))
        return im.size
    except Exception:
        return None

def figure_junk_reason(row, dims=None):
    w,h = dims or get_dim_from_record(row)
    if w and h:
        mn=min(w,h); mx=max(w,h); area=w*h; aspect=mx/max(1,mn)
        if mn <= MIN_DIMENSION_EXCLUDE:
            return 'min_dimension_too_small_header_footer_strip', {'width':w,'height':h,'area':area,'aspect':aspect}
        if area <= MIN_AREA_EXCLUDE:
            return 'area_too_small', {'width':w,'height':h,'area':area,'aspect':aspect}
        if aspect >= MAX_ASPECT_RATIO_EXCLUDE and mn <= 180:
            return 'extreme_aspect_ratio_strip', {'width':w,'height':h,'area':area,'aspect':aspect}
    text=' '.join(str(row.get(k,'')) for k in ['caption','label','title','description','ocr_text','alt_text'])[:1000]
    if CAPTION_OR_LABEL_KEYWORDS.search(text):
        return 'caption_label_header_footer_text_pattern', {'width':w,'height':h,'text_excerpt':text[:200]}
    return None, {'width':w,'height':h}

rows=list(iter_jsonl(local['figure_map']))
print('Figure map rows:', len(rows))

# First pass: metadata dimensions/text.
base_decisions=[]
need_dim=[]
for i,row in enumerate(rows):
    reason,info=figure_junk_reason(row)
    if reason:
        base_decisions.append((i, True, reason, info))
    else:
        w,h=get_dim_from_record(row)
        if DIMENSION_CHECK_IMAGES and Image and requests and (not w or not h):
            need_dim.append(i)
        base_decisions.append((i, False, None, info))

if MAX_DIMENSION_CHECK is not None:
    need_dim=need_dim[:MAX_DIMENSION_CHECK]
print('Need actual image dimension checks:', len(need_dim))

actual_dims={}
if DIMENSION_CHECK_IMAGES and need_dim:
    def work(i):
        url=image_url_for(rows[i])
        if not url: return i, None
        return i, read_http_dimensions(url)
    with concurrent.futures.ThreadPoolExecutor(max_workers=FIGURE_WORKERS) as ex:
        for n,(i,dim) in enumerate(ex.map(work, need_dim),1):
            if dim: actual_dims[i]=dim
            if n % 1000 == 0: print('dimension checked', n, 'dims found', len(actual_dims))

kept=[]; excluded=[]; final_decisions=[]
for i,row in enumerate(rows):
    reason,info=figure_junk_reason(row, actual_dims.get(i))
    if reason:
        rec={'row_index':i,'reason':reason,'info':info,'record_id':row.get('record_id') or row.get('figure_id') or row.get('id'), 'image_url':image_url_for(row), 'source_id':row.get('source_id'), 'page':row.get('page')}
        excluded.append(rec)
    else:
        kept.append(row)

filtered_path = STAGED/'textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN_GOVERNED_NO_JUNK_v10_5.jsonl'
write_jsonl(filtered_path, kept)
write_jsonl(OUT/'textbook_junk_figure_exclusions_STRICT_v10_5.jsonl', excluded)
fig_audit={'schema_version':'textbook_junk_figure_filter_audit.v10_5_strict','generated_at_utc':now(),'rules':{'min_dimension':MIN_DIMENSION_EXCLUDE,'min_area':MIN_AREA_EXCLUDE,'max_aspect_ratio':MAX_ASPECT_RATIO_EXCLUDE,'dimension_check_images':DIMENSION_CHECK_IMAGES,'actual_dims_found':len(actual_dims)},'counts':{'total':len(rows),'kept':len(kept),'excluded':len(excluded)},'examples_excluded':excluded[:50]}
write_json(OUT/'TEXTBOOK_JUNK_FIGURE_FILTER_AUDIT_STRICT_v10_5.json', fig_audit)
print(json.dumps(fig_audit, indent=2)[:6000])


In [ ]:
# Cell 7 — Upload staged/audit files and promote live objects
# Upload audits first.
uploads = {
    OUT/'LIVE_GOVERNED_METADATA_SCAN_v10_5.json': f"{GCS['audit_root']}/LIVE_GOVERNED_METADATA_SCAN_v10_5.json",
    OUT/'TEXTBOOK_JUNK_FIGURE_FILTER_AUDIT_STRICT_v10_5.json': f"{GCS['audit_root']}/TEXTBOOK_JUNK_FIGURE_FILTER_AUDIT_STRICT_v10_5.json",
    OUT/'textbook_junk_figure_exclusions_STRICT_v10_5.jsonl': f"{GCS['audit_root']}/textbook_junk_figure_exclusions_STRICT_v10_5.jsonl",
    STAGED/'textbook_lean_vector_manifest_v10_5_clean_summary.json': f"{GCS['stage_root']}/textbooks/textbook_lean_vector_manifest_v10_5_clean_summary.json",
    STAGED/'pathout_ap_diagnostic_vector_manifest_v10_5_clean_summary.json': f"{GCS['stage_root']}/pathout/pathout_ap_diagnostic_vector_manifest_v10_5_clean_summary.json",
    STAGED/'lecture_timecoded_vector_manifest_STRICT_CYTO_v10_5_clean_summary.json': f"{GCS['stage_root']}/lectures/lecture_timecoded_vector_manifest_STRICT_CYTO_v10_5_clean_summary.json",
    filtered_path: f"{GCS['stage_root']}/textbook_figures/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN_GOVERNED_NO_JUNK_v10_5.jsonl",
}
for src,dst in uploads.items():
    if Path(src).exists(): gcp(src, dst)

promotion={'schema_version':'pathology_hub_governed_cleanup_promotion.v10_5','generated_at_utc':now(),'promotion_mode':PROMOTION_MODE,'performed':False,'objects':[]}
if PROMOTION_MODE == 'backup_replace_live':
    replacements = [
        (GCS['tb_manifest'], STAGED/'textbook_lean_vector_manifest_v10_5_clean_summary.json', f"{GCS['backup_root']}/textbooks/textbook_lean_vector_manifest.json"),
        (GCS['pathout_manifest'], STAGED/'pathout_ap_diagnostic_vector_manifest_v10_5_clean_summary.json', f"{GCS['backup_root']}/pathout/pathout_ap_diagnostic_vector_manifest.json"),
        (GCS['lec_manifest'], STAGED/'lecture_timecoded_vector_manifest_STRICT_CYTO_v10_5_clean_summary.json', f"{GCS['backup_root']}/lectures/lecture_timecoded_vector_manifest_STRICT_CYTO_v9.json"),
        (GCS['figure_map'], filtered_path, f"{GCS['backup_root']}/textbook_figures/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"),
    ]
    for live, staged, backup in replacements:
        gcp(live, backup)
        gcp(staged, live)
        promotion['objects'].append({'live_uri':live,'backup_uri':backup,'promoted_from':str(staged)})
    promotion['performed']=True
else:
    promotion['notes']=['No promotion performed.']

if RESTART_CLOUD_RUN_AFTER_PROMOTION and promotion['performed']:
    p=sh(f"gcloud run services update {SERVICE} --region {REGION} --update-env-vars GOVERNANCE_V10_5_RESTART_TS={RUN_TS}", check=False, timeout=900)
    promotion['cloud_run_restart_requested']=True
    promotion['cloud_run_restart_returncode']=p.returncode
else:
    promotion['cloud_run_restart_requested']=False

write_json(OUT/'PATHOLOGY_HUB_GOVERNED_CLEANUP_PROMOTION_AUDIT_v10_5.json', promotion)
gcp(OUT/'PATHOLOGY_HUB_GOVERNED_CLEANUP_PROMOTION_AUDIT_v10_5.json', f"{GCS['audit_root']}/PATHOLOGY_HUB_GOVERNED_CLEANUP_PROMOTION_AUDIT_v10_5.json")
print(json.dumps(promotion, indent=2)[:8000])


In [ ]:
# Cell 8 — Health and authenticated API proof
import requests
proof={'schema_version':'pathology_hub_governed_cleanup_api_proof.v10_5','generated_at_utc':now(),'health':None,'search_tests':[]}
# Poll health after restart.
for i in range(20):
    try:
        r=requests.get(HEALTH_URL, timeout=30)
        proof['health']={'status_code':r.status_code,'text_excerpt':r.text[:3000]}
        if r.status_code == 200:
            try: proof['health_json']=r.json()
            except Exception: pass
            break
    except Exception as e:
        proof['health']={'error':str(e)}
    time.sleep(10)

payloads=[
    {"query":"melanoma invasive overview","sources":["lectures"],"max_results":5,"compact":True,"excerpt_char_limit":900},
    {"query":"ovarian high grade serous carcinoma p53 BRCA","sources":["who","textbooks","pathout","journals"],"max_results":5,"compact":True,"excerpt_char_limit":900},
    {"query":"prostate adenocarcinoma cribriform pattern 4","sources":["textbooks","pathout"],"max_results":5,"compact":True,"excerpt_char_limit":900},
]
headers={'Content-Type':'application/json'}
if API_KEY:
    headers['X-API-Key']=API_KEY
for payload in payloads:
    res={'payload':payload}
    try:
        r=requests.post(API_URL, headers=headers, json=payload, timeout=60)
        res['status_code']=r.status_code
        res['text_excerpt']=r.text[:3000]
        try:
            js=r.json(); res['json_keys']=list(js.keys())
            forbidden=[]
            def walk(x):
                if isinstance(x,dict):
                    t=x.get('primary_tag')
                    if t and is_forbidden(t): forbidden.append(t)
                    for v in x.values(): walk(v)
                elif isinstance(x,list):
                    for v in x: walk(v)
            walk(js)
            res['forbidden_returned_primary_tags']=sorted(set(forbidden))[:50]
            res['forbidden_count']=len(forbidden)
        except Exception as e: res['json_parse_error']=str(e)
    except Exception as e:
        res['error']=str(e)
    proof['search_tests'].append(res)

write_json(OUT/'PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5.json', proof)
gcp(OUT/'PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5.json', f"{GCS['audit_root']}/PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5.json")
print(json.dumps(proof, indent=2)[:12000])
if RUN_API_PROOF and API_KEY:
    bad=[t for t in proof['search_tests'] if t.get('status_code') != 200 or t.get('forbidden_count',0)>0]
    if bad:
        raise RuntimeError('API proof failed or returned forbidden primary tags. See API proof JSON.')
elif RUN_API_PROOF and not API_KEY:
    print('WARNING: No API key loaded; POST proof cannot be considered complete.')


In [ ]:
# Cell 9 — Package outputs for download
summary={
    'schema_version':'pathology_hub_governed_cleanup_v10_5_summary',
    'generated_at_utc':now(),
    'promotion_mode':PROMOTION_MODE,
    'promotion_performed':promotion.get('performed'),
    'figure_filter_counts':fig_audit['counts'],
    'metadata_scan_counts':{k:{'counts':v['counts'],'unique_primary_tags':v['unique_primary_tags']} for k,v in scans.items()},
    'api_key_loaded':bool(API_KEY),
    'api_proof_status_codes':[t.get('status_code') for t in proof.get('search_tests',[])],
    'notes':['v10.5 cleans stale manifest tag summaries, runs authenticated API proof if X-API-Key is present, and applies strict served-figure filtering.']
}
write_json(OUT/'PATHOLOGY_HUB_GOVERNED_CLEANUP_RUN_SUMMARY_v10_5.json', summary)
# Checksums
manifest={}
for p in list(OUT.rglob('*'))+list(STAGED.rglob('*')):
    if p.is_file():
        h=hashlib.sha256(p.read_bytes()).hexdigest()
        manifest[str(p.relative_to(WORK))]={'sha256':h,'size_bytes':p.stat().st_size}
write_json(OUT/'SHA256_MANIFEST_v10_5.json', manifest)
zip_path=WORK/'PATHOLOGY_HUB_GOVERNED_CLEANUP_V10_5_OUTPUTS.zip'
sh(f"cd {WORK} && zip -qr {zip_path} output staged")
print('Output ZIP:', zip_path)
if IN_COLAB:
    files.download(str(zip_path))
